# Heston Model — American Options via Monte Carlo (Longstaff–Schwartz)

Prices American options under the Heston stochastic-volatility model by Monte Carlo. 

The calculation has two stages:

| Stage | Method | Notes |
|---|---|---|
| **1. Path simulation** | log-Euler for $S$, full-truncation Euler for $v$ | correlated Brownian increments via the Cholesky factor of the $2\times2$ correlation matrix; the $\max(v, 0)$ truncation keeps variance non-negative |
| **2. American pricing** | Longstaff–Schwartz least-squares Monte Carlo (LSM) | backward induction: on in-the-money paths, regress the discounted continuation value on a polynomial in $S_t$, and exercise where intrinsic value exceeds the fitted continuation value |

Monte Carlo with LSM is the natural choice when the state space is high-dimensional or path-dependent: it handles the early-exercise feature without building a full PDE grid (like in case of the ADI method), at the cost of statistical (sampling) error rather than discretization error.


In [1]:
# Heston model American option pricing with Monte Carlo

import numpy as np
from tqdm import tqdm # For a progress bar

# --- 1. Heston SDE Simulation ---

def simulate_heston_paths(S0, v0, r, q, kappa, theta, sigma, rho, T, N, P, seed=None):
    """
    Simulates asset and variance paths using the Heston model.

    Parameters:
    S0 (float): Initial stock price
    v0 (float): Initial variance
    r (float): Risk-free interest rate
    q (float): Dividend yield
    kappa (float): Mean-reversion speed of variance
    theta (float): Long-term mean of variance
    sigma (float): Volatility of variance ('vol of vol')
    rho (float): Correlation between asset and variance processes
    T (float): Time to maturity in years
    N (int): Number of time steps
    P (int): Number of simulation paths

    Returns:
    tuple: A tuple containing two numpy arrays (S, v) for stock prices and variances.
    """

    dt = T / N
    # Initialize arrays for S and V paths
    S = np.zeros((N + 1, P))
    v = np.zeros((N + 1, P))
    S[0] = S0
    v[0] = v0
    
    # Generate correlated random numbers
    corr_matrix = np.array([[1, rho], [rho, 1]])
    L = np.linalg.cholesky(corr_matrix)

    rng = np.random.default_rng(seed)          # local, reproducible RNG
    
    for t in range(1, N + 1):
        # Generate two independent standard normal random variables
        Z = rng.standard_normal((2, P))
        #Z = np.random.standard_normal((2, P))
        # Correlate them using Cholesky decomposition
        corr_Z = L @ Z
        
        # Euler-Maruyama for S, Milstein for v (truncated at zero)
        # Discretize the stock price process
        S[t] = S[t-1] * np.exp((r - q - 0.5 * v[t-1]) * dt + 
                               np.sqrt(v[t-1]) * np.sqrt(dt) * corr_Z[0])
        # Discretize the variance process (Full Truncation scheme to prevent negative variance)
        v[t] = np.maximum(0, v[t-1] + kappa * (theta - v[t-1]) * dt + 
                                sigma * np.sqrt(v[t-1]) * np.sqrt(dt) * corr_Z[1])

    return S, v

In [2]:
# --- 2. Longstaff-Schwartz (LSM) Algorithm for American Option ---

def price_american_option_ls(S0, K, r, T, S, option_type='put', poly_degree=3):
    """
    Prices an American option using the Longstaff-Schwartz Monte Carlo method.

    Parameters:
    S0 (float): Initial stock price (used for discounting)
    K (float): Strike price
    r (float): Risk-free interest rate
    T (float): Time to maturity in years
    S (np.ndarray): Simulated stock price paths
    option_type (str): 'put' or 'call'
    poly_degree (int): Degree of the polynomial for regression

    Returns:
    float: Estimated price of the American option.
    """

    N, P = S.shape[0] - 1, S.shape[1]
    dt = T / N
    
    # Calculate intrinsic value at each time step
    if option_type == 'put':
        payoff = np.maximum(K - S, 0)
    else:
        payoff = np.maximum(S - K, 0)
        
    # Initialize cash flow matrix and exercise decision vector
    V_option = np.zeros_like(payoff)
    V_option[N, :] = payoff[N, :] # Payoff at maturity
    
    # Work backwards from T-1 to 1
    for t in range(N - 1, 0, -1):

        # Find paths that are in-the-money
        ITM = np.where(payoff[t, :] > 0)[0]

        # Get relevant stock prices and future cash flows for ITM paths
        S_ITM = S[t, ITM]

        # Get the slice of all future cash flows for the ITM paths
        future_CF_slice = V_option[t+1:, ITM] 
        # Create discount factors for each future time step back to t
        discounts = np.exp(-r * dt * np.arange(1, N - t + 1))
        # Calculate continuation value via matrix multiplication and summation
        CF_value_disc = np.sum(future_CF_slice * discounts[:, np.newaxis], axis=0)

        if len(ITM) > 0:    

            # Regression: Regress C_value_disc on S_ITM using a polynomial basis

            # X = S_ITM
            # Use only the stock price S_t for the basis functions, as in original LSM
            # Basis = np.vstack([np.ones(len(X)), X, X**2]).T # 1, S, S^2
            # Solve for coefficients (Least Squares Regression)
            # coefs = np.linalg.lstsq(Basis, C_value_disc, rcond=None)[0]
            # Need to use a small epsilon to avoid singular matrix warning for very small Basis
            # coefs = np.linalg.lstsq(Basis, C_value_disc, rcond=1e-6)[0]
            # Estimate Continuation Value C(S_t)
            # Continuation_Value = Basis @ coefs

            # Perform least-squares regression: E[Y|X]
            # Y is the discounted future cash flow, X is the stock price
            x = S_ITM / K # Normalize the regressor (x ~ O(1)) for a well-conditioned polynomial fit
            regression_coeffs = np.polyfit(x, CF_value_disc, poly_degree)
            continuation_value = np.polyval(regression_coeffs, x)
            #regression_coeffs = np.polyfit(S_ITM, CF_value_disc, poly_degree)
            #continuation_value = np.polyval(regression_coeffs, S_ITM)

            # Get intrinsic value for ITM paths
            intrinsic_value = payoff[t, ITM]
            
            # Find paths where immediate exercise is optimal
            exercise_paths = ITM[intrinsic_value > continuation_value]
            
            # Update cash flows for exercised paths
            V_option[t, exercise_paths] = intrinsic_value[intrinsic_value > continuation_value]
            
            # Set all future cash flows to zero for paths exercised now
            V_option[t+1:, exercise_paths] = 0
            
    # The option price is the mean of all discounted cash flows
    # Each column in cash_flow has at most one non-zero value (the exercise payoff)
    discount_factors = np.exp(-r * dt * np.arange(N + 1))
    option_price = np.mean(np.sum(V_option * discount_factors[:, np.newaxis], axis=0))
    
    return option_price


In [3]:
# --- 3. Main function execution ---

if __name__ == "__main__":
    
    # Model & Market Parameters (chosen to be realistic)
    S0 = 100.0     # Initial stock price
    v0 = 0.04      # Initial variance (volatility = sqrt(0.04) = 20%)
    r = 0.05       # Risk-free rate (5%)
    q = 0.02       # Dividend yield (2%)
    kappa = 2.0    # Speed of mean reversion
    theta = 0.0625 # Long-term variance (volatility = sqrt(0.0625) = 25%)
    sigma = 0.5    # Volatility of variance
    rho = -0.7     # Correlation (leverage effect: vol tends to rise when prices fall)
    
    # Option Parameters
    K = 100.0      # Strike price
    T = 1.0        # Time to maturity (1 year)
    
    # Simulation Parameters
    N = 250        # Number of time steps
    P = 50000      # Number of simulation paths
    poly_degree = 3 # Degree of polynomial for regression
    
    print("Pricing American Put Option using Heston Model and Longstaff-Schwartz...")
    
    # 1. Simulate Heston Paths
    print(f"Simulating {P} paths with {N} steps...")
    S_paths, v_paths = simulate_heston_paths(S0, v0, r, q, kappa, theta, sigma, rho, T, N, P, seed=42)

    # 2. Price the option
    print("Applying Longstaff-Schwartz algorithm...")
    american_put_price = price_american_option_ls(S0, K, r, T, S_paths, option_type='put', poly_degree=poly_degree)
    
    print("-" * 30)
    print(f"Model Parameters:")
    print(f"  S0={S0}, v0={v0}, r={r}, q={q}, kappa={kappa}, theta={theta}, sigma={sigma}, rho={rho}")
    print(f"Option Parameters:")
    print(f"  K={K}, T={T}, Type=American Put")
    print(f"Simulation Parameters:")
    print(f"  Steps={N}, Paths={P}, Poly Degree={poly_degree}")
    print("-" * 30)
    print(f"Estimated American Put Option Price: ${american_put_price:.4f}")
    print("-" * 30)
    

Pricing American Put Option using Heston Model and Longstaff-Schwartz...
Simulating 50000 paths with 250 steps...
Applying Longstaff-Schwartz algorithm...
------------------------------
Model Parameters:
  S0=100.0, v0=0.04, r=0.05, q=0.02, kappa=2.0, theta=0.0625, sigma=0.5, rho=-0.7
Option Parameters:
  K=100.0, T=1.0, Type=American Put
Simulation Parameters:
  Steps=250, Paths=50000, Poly Degree=3
------------------------------
Estimated American Put Option Price: $7.2293
------------------------------
